# Visión por Computador - Práctica 2 - Procesado de Imágenes

#### 10 puntos   |   Fecha de entrega: 15 de Noviembre de 2026, 23:59   |   Forma de entrega: a través de la tarea creada en https://pradogrado2627.ugr.es/

### Estudiante: Jesús Muñoz Velasco



---

## Normas de desarrollo y entrega de la práctica

1. Para este trabajo, al igual que para los demás, se debe presentar código, resultados, discusión de los mismos, y presentación y análisis del trabajo realizado, todo integrado en el propio Google Colab Notebook (es decir, no se entrega ninguna memoria separada en `.pdf` ni ningún código `.py`). Se recuerda que código y resultados sin informe explicativo no puntúa. Resulta también fundamental recordar que solo se acepta la entrega de ficheros `.ipynb` (no resulta válido entregar un fichero `.py`).

2. En relación con el punto anterior, se recuerda que se proporciona un Notebook, denominado `Ejemplo_Otsu.ipynb`, que muestra un ejemplo de respuesta y solución a un potencial ejercicio. El propósito de este Notebook es servir de referencia al alumnado acerca de cómo responder los ejercicios planteados. Véase que, por ejemplo, en aras de diferenciar con facilidad enunciados y respuestas, estas últimas se presentan en otro color (azul, en ese caso).

3. El diseño de celdas del documento debe ser respetado. Es decir, no se puede modificar el orden de los ejercicios o alterar la estructura del Notebook, de forma que dificulte localizar los contenidos. Aunque sí se puede, por ejemplo, crear nuevas subsecciones dentro de las secciones ya existentes para estructurar y organizar con más claridad las respuestas, así como para incluir los experimentos extra que se deseen.

4. Tal y como se muestra en `Ejemplo_Otsu.ipynb`, no hay ningún problema en emplear técnicas de IA generativa (como ChatGPT o Gemini) para crear el código de los distintos ejercicios o para enriquecer el análisis y descripción del trabajo realizado (aunque, obvia decirlo, su uso no es obligatorio). Pero, en caso de usarse, sí se debe aclarar de qué modo se ha usado la IA, con qué objetivo, y se debe demostrar que se entiende perfectamente lo que se ha hecho.

5. Solo se entregará el fichero `.ipynb` (incorporando código, resultados, y explicación del trabajo realizado y los resultados obtenidos) y no se enviarán las imágenes empleadas. El path para la lectura de imágenes, o cualquier otro fichero de entrada, debe ser siempre “/content/drive/My Drive/images/nombre_fichero”.

6. El código deberá  presentarse adecuadamente comentado y con los resultados obtenidos en cada apartado, junto con la presentación del trabajo realizado y la discusión y análisis de los resultados obtenidos. Es importante reiterar que la entrega de código sin informe explicativo o valoraciones no puntúa. Por otro lado, el código debe ir comentado en comentarios en celdas de código, mientras que el análisis y discusión, tanto del trabajo realizado como de los resultados obtenidos, debe ir en celdas de texto.

Este trabajo de implementación tiene como objetivo principal **familiarizarse con el uso de filtros de convolución, así como del cálculo de las derivadas de una imagen y sus posibles aplicaciones**.




### Materiales de Apoyo

En caso de que el alumnado tenga dudas sobre Python, OpenCV o el uso de Google Colab, se recuerda que en PRADO hay materiales de apoyo (https://pradogrado2627.ugr.es/mod/folder/view.php?id=83733):
-  Introducción a Python: https://pradogrado2627.ugr.es/pluginfile.php/153447/mod_folder/content/0/1.%20Introduccion%20a%20Python.pdf
- Introducción a NumPy, Matplotlib y Scikit-learn: https://pradogrado2627.ugr.es/pluginfile.php/153447/mod_folder/content/0/2.%20Introducci%C3%B3n%20a%20Numpy%20Matplotlib%20y%20sklearn.pdf
- Notebook con toda la información anterior (y más cosas): https://pradogrado2627.ugr.es/pluginfile.php/153447/mod_folder/content/0/Guia_Python.ipynb
- Ejercicios y ejemplos de repaso: https://pradogrado2627.ugr.es/pluginfile.php/153447/mod_folder/content/0/3.%20Ejercicios%20y%20ejemplos%20de%20repaso.pdf
- Fundamentos de Python para lectura, visualización y manipulación de imágenes: https://pradogrado2627.ugr.es/pluginfile.php/153447/mod_folder/content/0/4.%20Fundamentos%20de%20Python%20para%20manipulacion%20de%20imagenes.pdf(y Notebook asociado: https://pradogrado2627.ugr.es/pluginfile.php/153447/mod_folder/content/0/Guia_fundamentos_imagenes.ipynb)
- Ideas sobre cómo redactar un documento académico-científico (útil para saber cómo documentar las respuestas a los ejercicios, el Notebook en su conjunto, y la memoria del proyecto final o de un TFG/TFM): https://pradogrado2627.ugr.es/pluginfile.php/153447/mod_folder/content/0/Algunas%20ideas%20sobre%20c%C3%B3mo%20elaborar%20un%20informe%20acad%C3%A9mico-cient%C3%ADfico.pdf
- Ejemplo de respuesta a un hipotético ejercicio sobre el algoritmo de umbralización de Otsu: https://pradogrado2627.ugr.es/pluginfile.php/153448/mod_folder/content/0/Ejemplo_Otsu.ipynb



---



## Funciones de apoyo y conexión con Google Drive

In [ ]:
# We start by getting access to the drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# We import the main potential modules to use
import os,sys
import cv2, numpy as np, math
from matplotlib import pyplot as plt
# change directory
%cd '/content/drive/My Drive'
#  the current directory
%pwd

# This allows to display images  and to save  them in cells
%matplotlib inline
# this definition allows to read files in a specific path
get_image = lambda route: os.path.join('/content/drive/My Drive/images', route)

Las funciones que se incluyen a continuación, se proporcionan como ayuda/apoyo para el estudiante. **En caso de que el alumnado prefiera utilizar otras funciones para lectura, normalización y visualización de imágenes, tiene total libertad para hacerlo, pero sí debe justificar y explicar razonadamente el motivo de dicho cambio y describir las modificaciones realizadas.**

Recuérdese que todas las imágenes a emplear durante las prácticas de todo el curso se pueden encontrar en el enlace de Google Drive indicado en PRADO: https://pradogrado2627.ugr.es/mod/url/view.php?id=83732 En concreto, https://drive.google.com/drive/folders/1UtvY8q6w1Cz9a9T9zxIqSweoRLgNwoW-

In [ ]:
'''
This function receives a string with the filename of the image to read,
and a flag indicating if we want to read it in color/RGB (flagColor=1) or gray level (flagColor=0)

Example of use:
im1=readIm(get_image('apple.jpg'),0)

'''
def readIm(filename, flagColor=1):
  # cv2 reads BGR format
  im=cv2.imread(filename)
  # change to  RGB and return the image
  if(flagColor):
    return cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
  # change from BGR to grayscale instead if flag is 0
  return cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)

In [ ]:
'''
This function receives an array of arbitrary real numbers (that could include even negative values),
and returns an 'image' in the range [0,1].
flag_GLOBAL allows the user to normalize the whole image (including all channels) or to normalize
each channel/band independently.
'''
def rangeDisplay01(im, flag_GLOBAL=True):
    im = im.astype(float)
    min_val = im.min()
    max_val = im.max()
    if max_val == min_val:
        return np.zeros_like(im)
    if flag_GLOBAL or im.ndim == 2:
        im = (im - min_val) / (max_val - min_val)
    else:
        # Band-wise normalization
        for band in range(im.shape[2]):
            min_val = im[:, :, band].min()
            max_val = im[:, :, band].max()
            if max_val == min_val:
                im[:, :, band] = 0
            else:
                im[:, :, band] = (im[:, :, band] - min_val) / (max_val - min_val)
        # Note: remember that, for plt.imshow with RGB data,
        # the valid range is [0..1] for floats and [0..255] for integers.
    return im

In [ ]:
'''
This function displays a single image, including a title, using a magnification factor.

Example of use:
im1=readIm(get_image('apple.jpg'),0)
displayIm(im1,'Manzana',0.5)

'''
def displayIm(im, title='Result',factor= 2):
  # First normalize range
  max=np.max(im)
  min=np.min(im)
  if min<0 or max>255:
    im=rangeDisplay01(im,flag_GLOBAL=True)
  if len(im.shape) == 3:
    # im es tribanda
    plt.imshow(im, cmap='jet')
  else:
    # im es monobanda
    plt.imshow(im, cmap='gray')
  figure_size = plt.gcf().get_size_inches()
  plt.gcf().set_size_inches(factor * figure_size)
  plt.title(title)
  plt.xticks([]), plt.yticks([]) # eliminamos numeración
  plt.show()

In [ ]:
'''
This function displays multiple images (including a title and using a magnification factor)
of equal size. The input to the function is a list of images.

Example of use:
im1=readIm(get_image('apple.jpg'),1)
images = [im1,im1,im1]

displayMI_ES(images, title='Manzanas',factor=1)

IMPORTANT NOTE: it is important to notice that this function stacks all images
and later call displayIm(). This implies that the normalization (rangeDisplay01()) is done
at the very end of the process. As a consequence, if the stacked images have different
range values they can be displayed in a "weird" way (with black regions (pixel value = 0)
looking grey). In case the visualization seems to be a problem, a possible solution
would be to create vim using vim.append(rangeDisplay01(...)). In this way, all images
would be on the same range of values (negative values around 0 (black),
0 values around 0.5 (grey), and positive values closer to 1 (white))

'''
def displayMI_ES(vim, title='Result',factor=2):
  # We set the maximun number of images to concatenate by columns
  maxcolumns=3
  # number of images to display
  numberim=len(vim)
  # The simplest case, one row: the maximum number of columns is larger than the images to stack
  # This is, by default, we put together the images column-wise
  if len(vim) < maxcolumns:
    out=vim[0]
    for item in range(1,len(vim)):
      #displayIm(out,str(item),1)
      out=np.hstack((out,vim[item]))

  # Else, if maxcolumns is smaller or equal than the number of images to stack
  else:
    # We check if all rows and columns are going to be 'busy', or if otherwise we need an extra-row
    if np.mod(len(vim),maxcolumns)== 0:
      maxrows=numberim//maxcolumns
    else:
      maxrows=numberim//maxcolumns+1

    # we build up the first block-row
    out=vim[0]
    for item in range(1,maxcolumns):
      out=np.hstack((out,vim[item]))
    # We build up the rest of block-rows
    for block in range(1,maxrows):
      #print(block)
      row=vim[block*maxcolumns]
      for item in range(1,maxcolumns):
        # We still have images to stack
        if block*maxcolumns+item < numberim:
          row=np.hstack((row,vim[block*maxcolumns+item]))
        # We do not have more images, and we have to fill with black
        else:
          row=np.hstack((row,np.zeros(vim[0].shape,dtype=vim[0].dtype)))
          #print(row.dtype)
          # if we don't include ',dtype=vim[0].dtype', np.zeros will include float
          # numbers in the matrix. This will make that the whole matrix will be
          # considered as floats, and values larger than 1 will be clipped to 1,
          # displaying an almost totally white image
      # add the new block-row
      out=np.vstack((out,row))
  return displayIm(out,title,factor)

In [ ]:
# EL ALUMNADO DEBE COPIAR AQUÍ SUS FUNCIONES DE DISCRETIZACIÓN DE KERNELS GAUSSIANOS
# (desarrolladas en último apartado de la Práctica 1):
# GaussFunc, GaussDeriv1Func, GaussDeriv2Func y gaussianMask1D





# <font color='blue'>**Ejercicio 1: Filtrado Gaussiano** (3 puntos)

<font color='blue'>En este ejercicio aprenderemos a aplicar las máscaras de convolución ya discretizadas en la práctica 1 (https://pradogrado2627.ugr.es/mod/folder/view.php?id=83734). También buscamos hacerlo de un modo eficiente, y es por ello que, en toda la práctica (y no solo en este ejercicio), se exige el uso de convoluciones 1D (separables).





## <font color='blue'>A) (1 punto)

<font color='blue'>En este apartado se debe leer una imagen en escala de grises (`'cat.bmp'`) y esta debe ser filtrada mostrando el resultado de la convolución con: a) una Gaussiana, b) la primera derivada de la Gaussiana en $X$, c) la primera derivada de la Gaussiana en $Y$, d) la segunda derivada de la Gaussiana en $X$, y e) la segunda derivada de la Gaussiana en $Y$. Para ello, hay que:
1. emplear los siguientes sigmas ($\{1, 2, 4, 8, 16\}$);
2. usar la función de OpenCV `sepFilter2D()` con las máscaras 1D calculadas a partir de las funciones desarrolladas en la práctica anterior;
3. mostrar todas las imágenes dentro del mismo canvas (para facilitar su comparación);
4. y analizar los resultados obtenidos y discutir si se corresponden con lo que se esperaría obtener en base a nuestro conocimiento de la teoría. En este sentido, por ejemplo, resulta clave explicar qué par de kernels se aplican en cada caso, en qué orden y por qué.




In [ ]:

"""
    Apply Gaussian filtering or Gaussian-derivative filtering to an image.

    The function must construct the appropriate 1D Gaussian kernels (Gaussian,
    first derivative, or second derivative) for the given value of sigma and
    combine them using separable filtering.

    The parameter `orders` specifies the derivative order along the X and Y
    directions, respectively.
"""

def my2DConv(im, sigma, orders):
  # A CUBRIR POR EL ALUMNADO

  if orders==[0,0]: # Suavizado Gaussiano
    # A CUBRIR POR EL ALUMNADO
  else:
    print('error in order of derivative')

# A CUBRIR POR EL ALUMNADO

displayMI_ES(vim0,'Gaussian Smoothing',2)
displayMI_ES(vim1,'First Derivative (Y)',2)
displayMI_ES(vim1_1,'First Derivative (X)',2)
displayMI_ES(vim2,'Second Derivative (Y)',2)
displayMI_ES(vim2_1,'Second Derivative (X)',2)




## <font color='blue'>B) (1 punto)

<font color='blue'>En este apartado hay que emplear la función `my2DConv()`, creada en el apartado anterior (y que, a su vez, emplea la función de OpenCV `sepFilter2D()`) con $\sigma=1.5$ para calcular y mostrar el gradiente (dos imágenes: magnitud y orientación) y la Laplaciana de la Gaussiana. Se debe explicar el trabajo realizado, incluyendo las expresiones matemáticas subyacentes y el análisis, al igual que en todos los ejercicios, relativo a si los resultados obtenidos se corresponden con lo esperado. Se emplea la misma imagen del apartado anterior: `cat.bmp`

In [ ]:
# https://pyimagesearch.com/2021/05/12/image-gradients-with-opencv-sobel-and-scharr/
# print(math.atan2(-255,0)*180/math.pi)
# print(math.pi/2)

"""
gradientIM applies my2Dconv to compute the derivatives in the x and y directions,
which are returned as dx and dy, respectively.
"""
def gradientIM(im,sigma, displayImages=False):
  # A CUBRIR POR EL ALUMNADO
  return dx,dy

"""
laplacianG uses calls to my2Dconv to compute and return the Laplacian of the Gaussian.
"""
def laplacianG(im,sigma):
  return # A CUBRIR POR EL ALUMNADO

# A CUBRIR POR EL ALUMNADO

displayIm(laplacianG(im,sigma),'Laplaciana')





## <font color='blue'>C) (1 punto)  

<font color='blue'>Implemente una función para realizar "a mano" la convolución separable (es decir, empleando kernels 1D). Dicha función debe recibir como entrada una imagen (con el _padding_ adecuado, en caso de que lo necesite, ya incorporado) y un kernel 1D, y debe proporcionar como salida la imagen convolucionada con el kernel en ambas direcciones.

<font color='blue'>En esencia, la idea es que `convolve2D(image, kx, ky)` llame internamente dos veces a `convolve1D(image,kernel)`, una para realizar la convolución por filas y otra por columnas. En relación al padding, el hecho de introducirlo, o no, es un elemento externo a la función de convolución (que lo único que hace es realizar la convolución separable encajando el kernel hasta el límite de la imagen de entrada). El resultado de `convolve2D(image, kx, ky)` debe ser una imagen del mismo tamaño de la imagen de entrada `image` (que, como decimos, se asume que ya incluye el padding necesario).

<font color='blue'>Una vez haya implementado la función, aplique un kernel de derivadas en $X$ a la imagen `cat.bmp` (usando los kernels de tamaño $5$, generados por nuestras propias funciones de discretización, y dejando que sea la función de creación del kernel la que determine el $\sigma$ adecuado), y compare el resultado con el de `cv2.Sobel` (en donde se debe optar por la misma estrategia: fijando $ksize=5$, de modo que OpenCV construya automáticamente los kernels 1D correspondientes al operador de Sobel de ese tamaño). ¿Hay diferencias (sea a nivel de resultado final o de tiempo de ejecución)? En caso afirmativo, ¿a qué se pueden deber?

<font color='blue'>**Nota1**: el código del alumnado no puede ser más de $25$ veces más lento que el de OpenCV. Ese es el límite superior aceptable. Se debe optimizar la ejecución para, al menos, estar en ese orden de magnitud comparativamente con el rendimiento ofrecido por OpenCV. Estos tiempos de ejecución se deben medir usando:

```python
starttime = time.perf_counter()
# CÓDIGOS DE CONVOLUCIÓN
duration = timedelta(seconds=time.perf_counter() - starttime)
```
<font color='blue'>En el fragmento de código anterior, en donde se indica cómo medir el tiempo, cuando hablamos de `CÓDIGOS DE CONVOLUCIÓN` nos referimos única y exclusivamente a la llamada a `convolve2D`.

<font color='blue'>**Nota2:** no se puede utilizar en todo el ejercicio ni `from numpy.lib.stride_tricks import sliding_window_view` ni `np.convolve`.


In [ ]:
def convolve1D(image,kernel):
    # A CUBRIR POR EL ALUMNADO
    return output

def convolve2D(image, kx, ky):
    # A CUBRIR POR EL ALUMNADO
    return output2

# <font color='blue'>**Ejercicio 2**:  Pirámides Gaussianas y Laplacianas (4.5 puntos)







## <font color='blue'>A) (1 punto)

<font color='blue'>Construya una pirámide Gaussiana de $4$ niveles empleando máscaras Gaussianas de tamaño $7\times7$. Recuérdese que, cuando hablamos de $x$ niveles, nos referimos a la imagen original más $x$ niveles adicionales. Esta convención se aplicará tanto en este ejercicio como en el resto de la práctica. No se permite utilizar las funciones `pyrUp()` ni `pyrDown()` de OpenCV. Asimismo, `cv2.resize()` no podrá emplearse para construir la pirámide Gaussiana en el apartado A, aunque sí podrá utilizarse para construir la pirámide Laplaciana en el apartado B y para reconstruir la imagen original a partir de esta última en el apartado C. En todos los apartados de este ejercicio deberá utilizarse la imagen `motorcycle.bmp`. </font>

In [ ]:
'''
This function displays a pyramid of images, providing as input the list of images to display.

Example of use:
im1 = readIm(get_image('apple.jpg'), 1)
images = [
    im1,
    cv2.resize(im1, (im1.shape[1]//2,  im1.shape[0]//2),  interpolation=cv2.INTER_LINEAR),
    cv2.resize(im1, (im1.shape[1]//4,  im1.shape[0]//4),  interpolation=cv2.INTER_LINEAR),
    cv2.resize(im1, (im1.shape[1]//8,  im1.shape[0]//8),  interpolation=cv2.INTER_LINEAR),
    cv2.resize(im1, (im1.shape[1]//16, im1.shape[0]//16), interpolation=cv2.INTER_LINEAR)
]
displayPyramid(images, title='Piramide de Manzanas')
'''
def displayPyramid(vim, title='result'):
  temp = np.zeros_like(vim[0])
  if np.mod(temp.shape[1],2)==0:
    h = temp[:, :temp.shape[1]//2]
  else:
    h = temp[:, :(temp.shape[1]//2)+1]
  out = np.hstack((temp, h))
  top = 0
  out[0:vim[0].shape[0], 0:vim[0].shape[1]] = vim[0]
  left = vim[0].shape[1]
  for i in range(1, len(vim)):
    out[top:top+vim[i].shape[0], left:left + vim[i].shape[1]] = vim[i]
    top += vim[i].shape[0]
  displayIm(out, title)

In [ ]:
def pyramidGauss(im,sizeMask=7, nlevel=4):
  # A CUBRIR POR EL ALUMNADO
  return vim

# A CUBRIR POR EL ALUMNADO
displayPyramid(pyrG,'Gaussian Pyramid')


## <font color='blue'>B) (1 punto)

<font color='blue'>Use su código del apartado anterior para implementar una función que permita construir la pirámide Laplaciana asociada. Junto con la pirámide, debe mostrarse, para cada nivel, las imágenes a las que se les calcula la diferencia y el resultado de dicha diferencia.

In [ ]:
def pyramidLap(im, sizeMask,nlevel,flagInterpLap):
  # A CUBRIR POR EL ALUMNADO



## <font color='blue'>C) (1 punto)

<font color='blue'>Implemente ahora una función para reconstruir la imagen codificada en la pirámide Laplaciana del apartado anterior. Verifique que es posible obtener una copia exacta de la imagen original. Para ello, calcule el error de reconstrucción como el _Root Mean Squared Error_ (RMSE) entre los niveles de intensidad de la imagen original y la imagen reconstruida (redondeando al entero más cercano: `np.round()`).

In [ ]:
def reconstructLap(pyL,flagInterpReconstruct):
  # A CUBRIR POR EL ALUMNADO
  return im

# A CUBRIR POR EL ALUMNADO


print('Error per pixel',np.sqrt(np.mean((im - np.round(recons)) ** 2)))


## <font color='blue'>D) (1.5 puntos)



### <font color='blue'>APARTADO 1 (0.5 puntos)

<font color='blue'>Imaginemos ahora que no utilizamos el mismo procedimiento de expansión al construir la pirámide Laplaciana y al reconstruir la imagen original.En el primer caso, empleamos `cv2.resize()`, con `cv2.INTER_LINEAR`, mientras que en el segundo utilizamos `cv2.pyrUp()`. ¿Afecta esta diferencia al proceso de reconstrucción? ¿Por qué?

<font color='blue'>Se recomienda que, junto con la discusión de la respuesta y el razonamiento asociado, tanto en este apartado como en los siguientes, se aporte evidencia empírica que permita verificar/apoyar la hipótesis planteada por el alumno. Por ejemplo, puede desarrollarse un pequeño script que implemente la modificación propuesta en el enunciado y permita analizar experimentalmente sus efectos.

### <font color='blue'>APARTADO 2 (0.5 puntos)

<font color='blue'>¿Es posible reconstruir a la perfección (`Error per pixel 0.0`) la imagen original por medio de una pirámide Laplaciana construida a partir de una pirámide Gaussiana sin suavizado (es decir, una pirámide no realmente Gaussiana, construida exclusivamente en base a hacer subsampling)?

### <font color='blue'>APARTADO 3 (0.5 puntos)

<font color='blue'>Intentemos ahora crear la pirámide Laplaciana de "abajo a arriba". Es decir, tal y como hemos visto en teoría, la pirámide Gaussiana se construye de "abajo a arriba", y la Laplaciana se construye de "arriba a abajo" (desde la cima de la pirámide hacia la base). Pero, ¿qué pasa si la Laplaciana también se construye de "abajo a arriba"? Hablamos de partir de la imagen original y construir la Gaussiana y la Laplaciana al mismo tiempo, de "abajo a arriba" simultáneamente:
1) Suavizo la imagen original, calculo la resta de la original y su suavizada. Y con eso ya tengo el "nivel de abajo" de la pirámide Laplaciana.
2) Submuestreo la imagen original suavizada (y ya tengo el siguiente nivel de la pirámide Gaussiana). Resto esa imagen a una versión suavizada de la misma. Y con eso ya tengo el siguiente nivel de la pirámide Laplaciana.
3) Así sucesivamente.

<font color='blue'>En otras palabras: en lugar de definir cada nivel como la diferencia entre $G_i$ y la expansión del siguiente nivel gaussiano $G_{i+1}$, calculamos directamente la diferencia entre $G_i$ y su versión suavizada $H(G_i)$. Después, submuestreamos $H(G_i)$ para obtener $G_{i+1}$ y repetimos el proceso. ¿Esto sería un proceso válido para construir la pirámide Laplaciana? ¿Nos seguiría permitiendo reconstruir la imagen de entrada?



# <font color='blue'>**Ejercicio 3: Imágenes Híbridas** (1.25 puntos)

<font color='blue'>Este ejercicio se basa en el siguiente trabajo: $\textit{Oliva, A., Torralba, A., & Schyns, P. G. (2006). Hybrid images. ACM Transactions on Graphics (TOG), 25(3), 527-532}$ (https://stanford.edu/class/ee367/reading/OlivaTorralb_Hybrid_Siggraph06.pdf).

<font color='blue'>Vamos a profundizar un poco en cómo el sistema visual humano extrae información sobre un objeto en función de la distancia. Para ello, construimos una imagen híbrida a partir de dos imágenes de objetos diferentes. Mezclando adecuadamente parte de las altas frecuencias de una imagen con parte de las bajas frecuencias de la otra, obtenemos una imagen híbrida que muestra diferentes percepciones con la distancia.

<font color='blue'>El sigma empleado para filtrar ambas imágenes (tanto la de alta como la de bajas frecuencias) es el aspecto clave para seleccionar el rango de frecuencias altas y bajas de cada imagen. Cuanto mayor sea el valor sigma, mayor será la eliminación de altas frecuencias de la imagen. Se recomienda elegir este valor por separado para cada una de las imágenes y, de hecho, es deseable tener distintos valores para distintos pares de imágenes.

<font color='blue'>Aspectos de importancia en relación a este ejercicio:

1. Como mínimo, se deben generar las imágenes híbridas correspondientes a  `'cat.bmp'+'dog.bmp'`, `'submarine.bmp'+'fish.bmp'` y `'marilyn.bmp'+'einstein.bmp'` (todas en escala de grises). Y también el alumnado debe escoger, como mínimo, otro par de imágenes de su elección a hibridar en color, fuera de las incluidas en la carpeta compartida en Drive (https://drive.google.com/drive/folders/1UtvY8q6w1Cz9a9T9zxIqSweoRLgNwoW-). Es decir, como mínimo, debe haber $4$ pares de imágenes híbridas como parte de la respuesta a este ejercicio. Se debe emplear una pirámide Gaussiana con $6$ niveles para mostrar el efecto obtenido.

<font color='blue'>**Nota:** de cara a emplear imágenes que no estén en el repositorio compartido, se recomienda que el alumnado emplee un código como el que se muestra a continuación, y que permite descargar sobre la marcha imágenes a partir de una URL dada. En el caso del par de imágenes seleccionado por el alumnado, ambas imágenes deberán tener tamaños compatibles y estar razonablemente alineadas, especialmente en las estructuras que se pretende superponer.






In [ ]:
import requests
def downloadImage(url):
    response = requests.get(url)
    img_array = np.frombuffer(response.content, np.uint8)
    img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.astype(np.float64)/255
    return img
try:
    smiley_im = downloadImage("https://images.rawpixel.com/image_800/cHJpdmF0ZS9sci9pbWFnZXMvd2Vic2l0ZS8yMDIyLTA1L2ZsMTI5Mjk0MTAxNjUtaW1hZ2Uta3liZW9jNjEuanBn.jpg")
    plt.imshow(smiley_im)
except:
    print("Couldn't download the image from the given URL")

<font color='blue'>2. No es necesario seguir al pie de la letra el artículo de referencia y operar en el dominio de la frecuencia (Fourier). Basta con operar en el dominio del espacio, como hemos hecho en todos los demás apartados de esta práctica. No obstante, sí se debe emplear/buscar algún criterio objetivable/cuantificable a la hora de escoger qué imagen se empleará para las altas frecuencias y cuál para las bajas frecuencias. No basta con escogerla "a ojo" y, al contrario, se debe definir alguna métrica que permita realizar la selección.

<font color='blue'>3. En relación con el punto anterior, se debe explicar con claridad el proceso seguido para filtrar la imagen de altas frecuencias y la de bajas frecuencias, incluyendo la elección de los sigmas finalmente utilizados. La realización de una experimentación preliminar, por medio de prueba y error, es perfectamente válida, pero hay que explicarla y justificarla de algún modo. No basta con probar una serie de valores arbitrarios y justificarlo, sucintamente, indicando que *"se procedió por medio de prueba y error hasta obtener el sigma final"*.

In [ ]:
# A CUBRIR POR EL ALUMNADO

# <font color='blue'>**Ejercicio 4: Pyramid Blending** (1.25 puntos)

<font color='blue'>En este ejercicio se deberá construir una imagen fusionada a partir de un par de imágenes reales, correspondiente a las imágenes de la manzana y la naranja. Se debe crear la imagen fusionada de la manzana (`apple.jpg`; parte izquierda de la imagen fusionada final) y la naranja (`orange.jpg`; parte derecha de la imagen fusionada final).

<font color='blue'>No se podrán utilizar directamente las funciones `cv2.pyrDown()` ni `cv2.pyrUp()`. La construcción de las pirámides deberá realizarse haciendo uso de las operaciones de suavizado, submuestreo e interpolación implementadas previamente en la práctica.

<font color='blue'>Se deberá:

1. <font color='blue'>Explicar detalladamente el procedimiento seguido para obtener la imagen fusionada, incluyendo la construcción de las pirámides gaussianas y laplacianas, la mezcla de los distintos niveles y la posterior reconstrucción de la imagen final.

2. <font color='blue'>Comparar  el resultado obtenido mediante pyramid blending con dos alternativas más sencillas:
   - <font color='blue'>una combinación directa, sin blending, formada tomando los píxeles de la mitad izquierda de una imagen y los de la mitad derecha de la otra;
   - una combinación a una única escala, empleando una sola máscara binaria suavizada mediante un filtro gaussiano (con $\sigma$ = $2$, $5$ y $15$) para mezclar directamente ambas imágenes a resolución completa.

3. <font color='blue'>Visualizar los principales estados intermedios del proceso. En particular, se deberán mostrar:
   - <font color='blue'>las pirámides gaussianas (de la manzana, la naranja y la máscara binaria) con seis niveles cada una;
   - las pirámides laplacianas correspondientes (de la manzana y la naranja);
   - y la laplaciana fusionada con los distintos niveles resultantes de la mezcla.

4. <font color='blue'>Finalmente, se deberá responder razonadamente a la siguiente cuestión:
> <font color='blue'>¿En qué situaciones resulta ventajoso emplear pyramid blending frente a utilizar una única máscara suavizada? ¿En qué casos podría ser suficiente una mezcla a una sola escala?




In [ ]:
# A CUBRIR POR EL ALUMNADO